# 00 — Setup and reference verification

Verifies the environment and the frozen `references/` snapshot. Produces no analysis.

- Python 3.12 (asserted, not assumed)
- every `references/` file read-only and sha256-matching `references/MANIFEST.sha256`
- the Aim 2 artefacts this project depends on are present

Idempotent. Run timestamps go to the gitignored `provenance.jsonl`, never into the
report body, so `notebook_00_sha256.json` is reproducible byte-for-byte.

## 1 — Header

In [1]:
import sys, random, time, warnings
from pathlib import Path

import numpy as np

random.seed(42)
np.random.seed(42)

_cwd = Path('.').resolve()
ROOT = _cwd if (_cwd / 'src').is_dir() else _cwd.parent
assert (ROOT / 'src').is_dir(), f"Cannot locate src/ -- tried {_cwd} and {_cwd.parent}"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print(f'ROOT: {ROOT}')

warnings.filterwarnings('ignore', category=FutureWarning)

import pandas as pd
from src.utils import log_versions, run_timestamp, record_provenance, write_report, write_sha256_manifest

RUN_TIMESTAMP = run_timestamp()
VERSIONS = log_versions(['pandas', 'numpy', 'scipy', 'sklearn', 'matplotlib'])
print(f'Run: {RUN_TIMESTAMP}')
assert sys.version_info[:2] == (3, 12), f'Expected Python 3.12, got {sys.version_info[:2]}'
print(f'Python: {sys.version.split()[0]}')

ROOT: /Users/rinatrizvanov/Desktop/Projects/Project 1 - Comp Bio for Osteosarcoma/tcab1-bir-dependency


Library versions:
  pandas: 2.3.3
  numpy: 2.5.4
  scipy: 1.18.1
  sklearn: 1.9.1
  matplotlib: 3.11.2
Run: 2026-10-12T01:23:11Z
Python: 3.12.13


## 2 — Reference snapshot integrity

In [2]:
from src.utils import REFERENCES_DIR, sha256_file
import os, stat

manifest_lines = (REFERENCES_DIR / 'MANIFEST.sha256').read_text().splitlines()
expected = {}
for line in manifest_lines:
    if line.startswith('#') or not line.strip():
        continue
    digest, name = line.split(maxsplit=1)
    expected[name.strip()] = digest

# git tracks the executable bit but NOT the write bit, so the chmod -w applied
# when references/ was created does not survive `git clone`. This notebook is
# the setup step, so it RESTORES the read-only mode rather than asserting an
# invariant that cloning necessarily breaks. The sha256 check below stays a hard
# assertion -- that is the integrity guarantee that actually matters.
restored = []
for name in sorted(expected):
    path = REFERENCES_DIR / name
    if path.exists() and (path.stat().st_mode & 0o222):
        os.chmod(path, stat.S_IRUSR | stat.S_IRGRP | stat.S_IROTH)
        restored.append(name)
if restored:
    print(f'Restored read-only mode on {len(restored)} reference file(s) '
          f'(expected after a fresh clone):')
    for name in restored:
        print(f'    {name}')
    print()

rows = []
for name, want in sorted(expected.items()):
    path = REFERENCES_DIR / name
    rows.append({
        'file': name,
        'exists': path.exists(),
        'sha256_match': sha256_file(path) == want if path.exists() else False,
        'mode': oct(path.stat().st_mode)[-3:] if path.exists() else '---',
        'read_only': not (path.stat().st_mode & 0o222) if path.exists() else False,
        'size_bytes': path.stat().st_size if path.exists() else 0,
    })
refs = pd.DataFrame(rows)
print(refs.to_string(index=False))

assert refs['exists'].all(), f"Missing reference files: {refs.loc[~refs['exists'], 'file'].tolist()}"
assert refs['sha256_match'].all(), f"Checksum drift: {refs.loc[~refs['sha256_match'], 'file'].tolist()}"
assert refs['read_only'].all(), f"Could not restore read-only: {refs.loc[~refs['read_only'], 'file'].tolist()}"
print(f"\nAll {len(refs)} reference files verified, read-only, checksums match.")
if restored:
    print(f"({len(restored)} had write permission restored to 444 by this notebook.)")

                        file  exists  sha256_match mode  read_only  size_bytes
           aim2_signature.py    True          True  444       True       28320
               aim2_utils.py    True          True  444       True        5380
feasibility_decision_memo.md    True          True  444       True        8293
              label_rules.py    True          True  444       True       16708
          label_spec_v1.0.md    True          True  444       True        5718
         manifest_builder.py    True          True  444       True       35068
         master_manifest.csv    True          True  444       True       64029
       notebook_13_report.md    True          True  444       True        4432

All 8 reference files verified, read-only, checksums match.


## 3 — Aim 2 provenance recorded in the snapshot

In [3]:
header = [l for l in manifest_lines if l.startswith('#')]
for line in header[:14]:
    print(line)

# references/ — frozen read-only copies from the Aim 2 project
#
# Source project : /Users/rinatrizvanov/Desktop/Projects/Project 1 - Comp Bio for Osteosarcoma/tcab1-alt-osteo pipeline
# Source git HEAD: 5af4d0f31ed0d777c2daca32c4cfd1e2d8d02ea8
# Source git log : 5af4d0f 2026-06-12 notebooks 13
# Copied         : 2026-10-11T22:54:32Z
# Mode           : all files chmod -w (read-only) after copy
#
# master_manifest.csv is the FROZEN LABEL INPUT for Stage 1.
# Stage 1 never re-derives ALT labels; it reads alt_call_direction / alt_status from here.
#
# Original paths within the source project:
#   label_spec_v1.0.md           <- docs/label_spec_v1.0.md
#   feasibility_decision_memo.md <- docs/feasibility_decision_memo.md


## 4 — Report

In [4]:
lines = [
    '# Notebook 00 — Setup and reference verification',
    '',
    'Verifies the environment and the frozen `references/` snapshot. No analysis.',
    '',
    '## Environment',
    '',
    f"- Python: {sys.version.split()[0]}",
    '',
    '| Library | Version |',
    '|---|---|',
]
lines += [f'| {k} | {v} |' for k, v in VERSIONS.items()]
lines += [
    '',
    '## Reference snapshot',
    '',
    f'All {len(refs)} files present, read-only (mode 444), sha256 matching `references/MANIFEST.sha256`.',
    '',
    ('Note: git does not track the write bit, so `chmod -w` does not survive `git clone`. '
     'This notebook restores mode 444 when it finds a reference file writable; the sha256 '
     'check is the hard integrity guarantee. Whether anything needed restoring on a given '
     'run is recorded in `provenance.jsonl`, not here, so this report stays byte-identical '
     'across runs.'),
    '',
    '| File | sha256 match | Mode | Bytes |',
    '|---|---|---|---|',
]
for _, r in refs.iterrows():
    lines.append(f"| {r['file']} | {'yes' if r['sha256_match'] else 'NO'} | {r['mode']} | {r['size_bytes']} |")
lines += [
    '',
    '## Source provenance',
    '',
    '```',
] + [l.lstrip('# ').rstrip() for l in header if 'Source' in l or 'Copied' in l] + [
    '```',
    '',
    '`master_manifest.csv` is the frozen label input. Stage 1 never re-derives ALT labels.',
]

report = write_report('00', lines)
manifest = write_sha256_manifest('00', [report])
record_provenance([{
    'notebook': '00', 'run_timestamp': RUN_TIMESTAMP,
    'python': sys.version.split()[0], 'versions': VERSIONS,
    'n_reference_files': int(len(refs)), 'all_verified': True,
    'read_only_restored': restored,
}])
print(f'wrote {report.name}')
print(f'wrote {manifest.name}')

wrote notebook_00_report.md
wrote notebook_00_sha256.json
